# Stage 2.2 — Multi-Scale Neural Ensemble for Trade Signal Forecasting

**Hypothesis:** trade signals have different predictive power at different time horizons. A model trained on weekly patterns sees different alpha than one trained on monthly or quarterly patterns.

| Model | Architecture | Lookback | Captures |
|-------|-------------|----------|----------|
| **MLP** | 3-layer feedforward | 0 (cross-sectional) | Contemporaneous factor exposure |
| **Transformer** | 4-head encoder | 20 days | Short-term attention patterns |
| **NN-Weekly** | LSTM | 5 days | Weekly momentum / reversals |
| **NN-Monthly** | LSTM | 21 days | Monthly trend / carry |
| **NN-Quarterly** | LSTM | 63 days | Quarterly regime / factor rotation |
| **MetaLearner** | Ridge on all above | — | IC-weighted combination |

**Reads from Stage 2:** `master_dataset.parquet`, `alpha_mu.parquet`  
**Writes:** `alpha_mu_neural.parquet` (drops into Stage 3 alongside existing models)

## 0. Imports & Config

In [1]:
import os, json, time, warnings
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from IPython.display import display
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

# ── Paths ─────────────────────────────────────────────────────────────────────
WORKSPACE  = '/Applications/Projects/Hackathons/quantum_ai/workspace'
DATA_DIR   = os.path.join(WORKSPACE, 'data')
MODELS_DIR = os.path.join(WORKSPACE, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

TRAIN_END  = '2017-12-31'
TEST_START = '2018-01-01'
TARGET     = 'fwd_return_5d'

# ── Device ────────────────────────────────────────────────────────────────────
if torch.backends.mps.is_available() and torch.backends.mps.is_built():
    DEVICE = 'mps'
elif torch.cuda.is_available():
    DEVICE = 'cuda'
else:
    DEVICE = 'cpu'

# ── Training knobs ────────────────────────────────────────────────────────────
MLP_EPOCHS         = 50
TRANSFORMER_EPOCHS = 40
SCALE_EPOCHS       = 35       # per scale NN
BATCH_SIZE         = 256
LR                 = 1e-3
DROPOUT            = 0.2
SEED               = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

print(f'Device : {DEVICE}')
print(f'PyTorch: {torch.__version__}')

Device : mps
PyTorch: 2.9.1


## 1. Load Data

In [2]:
master = pd.read_parquet(os.path.join(DATA_DIR, 'master_dataset.parquet'))
master['timestamp'] = pd.to_datetime(master['timestamp'])
master = master.set_index(['timestamp', 'ticker'])

EXCLUDE = {
    'sector', 'price', 'high', 'low', 'volume',
    'return_1d', 'fwd_return_1d', 'fwd_return_5d', 'regime',
    'news_today', 'news_yesterday', 'market_news_today', 'market_news_yesterday',
    'n_news_today', 'n_market_news_today', 'news_sent_today', 'news_sent_yesterday',
}
SIGNAL_COLS = [c for c in master.columns if c not in EXCLUDE]
TICKERS     = master.index.get_level_values('ticker').unique().tolist()
N_FEATURES  = len(SIGNAL_COLS)

df = master.dropna(subset=[TARGET]).copy()
ts = df.index.get_level_values('timestamp')
train_mask = ts <= pd.Timestamp(TRAIN_END)
test_mask  = ts >= pd.Timestamp(TEST_START)

X_train = df.loc[train_mask, SIGNAL_COLS]
y_train = df.loc[train_mask, TARGET]
X_test  = df.loc[test_mask, SIGNAL_COLS]
y_test  = df.loc[test_mask, TARGET]

# Fill NaN with train medians
train_medians = X_train.median()
X_train_f = X_train.fillna(train_medians)
X_test_f  = X_test.fillna(train_medians)

# Standardise
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train_f)
X_test_s  = scaler.transform(X_test_f)

print(f'Train : {X_train_s.shape}  |  Test: {X_test_s.shape}')
print(f'Features : {N_FEATURES}  |  Tickers: {len(TICKERS)}')
print(f'Signal cols: {SIGNAL_COLS[:5]} ...')

Train : (25180, 28)  |  Test: (12180, 28)
Features : 28  |  Tickers: 20
Signal cols: ['mom_5d', 'mom_20d', 'mom_60d', 'rev_1d', 'vol_20d'] ...


## 2. Metric Helpers

In [3]:
def spearman_ic(pred: pd.Series, actual: pd.Series) -> float:
    d = pd.concat([pred, actual], axis=1).dropna()
    return float(spearmanr(d.iloc[:, 0], d.iloc[:, 1])[0]) if len(d) >= 10 else float("nan")


def long_short_sharpe(pred: pd.Series, actual: pd.Series, top_n: int = 5) -> float:
    d = pd.concat([pred.rename("p"), actual.rename("y")], axis=1).dropna().reset_index()
    daily = []
    for _, g in d.groupby("timestamp"):
        if len(g) < 2 * top_n:
            continue
        g = g.sort_values("p")
        daily.append(g.tail(top_n)["y"].mean() - g.head(top_n)["y"].mean())
    if not daily:
        return float("nan")
    r = pd.Series(daily)
    return float(r.mean() / (r.std() + 1e-9) * np.sqrt(252 / 5))


def score(name: str, pred: pd.Series, actual: pd.Series) -> dict:
    # align on shared index before any element-wise comparison
    pred, actual = pred.align(actual, join="inner")
    pred   = pred.dropna()
    actual = actual.reindex(pred.index)
    return {
        "model":   name,
        "IC":      spearman_ic(pred, actual),
        "dir_acc": float((np.sign(pred) == np.sign(actual)).mean()),
        "sharpe":  long_short_sharpe(pred, actual),
    }


print("Metrics ready.")

Metrics ready.


## 3. Model 1 — MLP Baseline

Cross-sectional (no time dimension): 28 signals → hidden layers → predicted return.  
Trained on all (date, ticker) rows pooled together.

In [4]:
class MLPForecaster(nn.Module):
    def __init__(self, n_in: int, hidden: list = [256, 128, 64]):
        super().__init__()
        layers = []
        dim = n_in
        for h in hidden:
            layers += [nn.Linear(dim, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(DROPOUT)]
            dim = h
        layers.append(nn.Linear(dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(-1)


def train_mlp(X_tr, y_tr, X_te, y_te_idx, epochs=MLP_EPOCHS):
    X_t = torch.tensor(X_tr, dtype=torch.float32)
    y_t = torch.tensor(y_tr.values, dtype=torch.float32)
    loader = DataLoader(TensorDataset(X_t, y_t), batch_size=BATCH_SIZE, shuffle=True)

    model = MLPForecaster(X_tr.shape[1]).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.HuberLoss(delta=0.01)

    print(f'Training MLP ({sum(p.numel() for p in model.parameters()):,} params) '
          f'for {epochs} epochs on {DEVICE}...')
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            ep_loss += loss.item()
        sched.step()
        if ep % 10 == 0:
            print(f'  ep {ep:3d}/{epochs}  loss={ep_loss/len(loader):.5f}')

    print(f'Done in {time.time()-t0:.1f}s')
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X_te, dtype=torch.float32).to(DEVICE)).cpu().numpy()
    return model, pd.Series(preds, index=y_te_idx, name='mlp')


mlp_model, mlp_pred = train_mlp(X_train_s, y_train, X_test_s, y_test.index)
mlp_scores = score('MLP', mlp_pred, y_test)
print(f'\nMLP  IC={mlp_scores["IC"]:+.4f}  dir_acc={mlp_scores["dir_acc"]:.3f}  sharpe={mlp_scores["sharpe"]:+.3f}')

Training MLP (49,537 params) for 50 epochs on mps...
  ep  10/50  loss=0.00016
  ep  20/50  loss=0.00016
  ep  30/50  loss=0.00015
  ep  40/50  loss=0.00015
  ep  50/50  loss=0.00015
Done in 27.4s

MLP  IC=+0.0581  dir_acc=0.554  sharpe=+0.907


## 4. Model 2 — Temporal Transformer

For each (ticker, test_date) we build a sequence of the last `LOOKBACK` days of features.  
A Transformer encoder with 4 attention heads reads the sequence; the last token's hidden state is projected to a return prediction.

In [5]:
TRANSFORMER_LOOKBACK = 20

def build_sequence_dataset(df_full, signal_cols, target, lookback, train_end, test_start, scaler):
    """Build (seq_len x features) windows per (ticker, date)."""
    # Work per ticker to preserve time order
    X_seqs_tr, y_tr_list, idx_tr = [], [], []
    X_seqs_te, y_te_list, idx_te = [], [], []

    feat_arr_full = scaler.transform(df_full[signal_cols].fillna(df_full[signal_cols].median()))
    feat_df = pd.DataFrame(feat_arr_full, index=df_full.index, columns=signal_cols)

    for ticker in df_full.index.get_level_values('ticker').unique():
        try:
            tkr_feat = feat_df.xs(ticker, level='ticker').sort_index()
            tkr_tgt  = df_full.xs(ticker, level='ticker')[target].sort_index().dropna()
        except KeyError:
            continue

        dates = tkr_tgt.index
        for i, date in enumerate(dates):
            if i < lookback:
                continue
            window_dates = tkr_feat.index[tkr_feat.index <= date][-lookback:]
            if len(window_dates) < lookback:
                continue
            seq = tkr_feat.loc[window_dates].values.astype(np.float32)  # (lookback, features)
            label = float(tkr_tgt[date])
            if date <= pd.Timestamp(train_end):
                X_seqs_tr.append(seq)
                y_tr_list.append(label)
                idx_tr.append((date, ticker))
            elif date >= pd.Timestamp(test_start):
                X_seqs_te.append(seq)
                y_te_list.append(label)
                idx_te.append((date, ticker))

    idx_tr = pd.MultiIndex.from_tuples(idx_tr, names=['timestamp', 'ticker'])
    idx_te = pd.MultiIndex.from_tuples(idx_te, names=['timestamp', 'ticker'])
    return (np.array(X_seqs_tr), np.array(y_tr_list), idx_tr,
            np.array(X_seqs_te), np.array(y_te_list), idx_te)


print(f'Building sequence dataset (lookback={TRANSFORMER_LOOKBACK})...')
t0 = time.time()
Xs_tr, ys_tr, idx_tr, Xs_te, ys_te, idx_te = build_sequence_dataset(
    df, SIGNAL_COLS, TARGET, TRANSFORMER_LOOKBACK, TRAIN_END, TEST_START, scaler
)
print(f'Done in {time.time()-t0:.1f}s  |  train={Xs_tr.shape}  test={Xs_te.shape}')

Building sequence dataset (lookback=20)...
Done in 3.3s  |  train=(24780, 20, 28)  test=(12180, 20, 28)


In [6]:
class TemporalTransformer(nn.Module):
    def __init__(self, n_features: int, d_model: int = 64, nhead: int = 4,
                 num_layers: int = 2, dropout: float = DROPOUT):
        super().__init__()
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_enc    = nn.Parameter(torch.zeros(1, 512, d_model))  # learnable positional
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, norm_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.head    = nn.Sequential(nn.LayerNorm(d_model), nn.Linear(d_model, 1))

    def forward(self, x):           # x: (B, T, F)
        T = x.size(1)
        x = self.input_proj(x) + self.pos_enc[:, :T, :]
        x = self.encoder(x)
        return self.head(x[:, -1, :]).squeeze(-1)  # use last token


def train_seq_model(model, X_tr, y_tr, X_te, idx_te, epochs, name='model'):
    Xt = torch.tensor(X_tr, dtype=torch.float32)
    yt = torch.tensor(y_tr, dtype=torch.float32)
    loader = DataLoader(TensorDataset(Xt, yt), batch_size=BATCH_SIZE, shuffle=True)

    model = model.to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(
        opt, max_lr=LR * 5, steps_per_epoch=len(loader), epochs=epochs
    )
    loss_fn = nn.HuberLoss(delta=0.01)

    print(f'Training {name} ({sum(p.numel() for p in model.parameters()):,} params) '
          f'for {epochs} epochs...')
    t0 = time.time()
    for ep in range(1, epochs + 1):
        model.train()
        ep_loss = 0
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
            ep_loss += loss.item()
        if ep % 10 == 0:
            print(f'  ep {ep:3d}/{epochs}  loss={ep_loss/len(loader):.5f}')

    print(f'Done in {time.time()-t0:.1f}s')
    model.eval()
    all_preds = []
    with torch.no_grad():
        for i in range(0, len(X_te), BATCH_SIZE):
            xb = torch.tensor(X_te[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            all_preds.append(model(xb).cpu().numpy())
    return model, pd.Series(np.concatenate(all_preds), index=idx_te, name=name)


transformer_model, transformer_pred = train_seq_model(
    TemporalTransformer(N_FEATURES),
    Xs_tr, ys_tr, Xs_te, idx_te,
    epochs=TRANSFORMER_EPOCHS, name='transformer'
)
transformer_scores = score('Transformer', transformer_pred, y_test)
print(f'\nTransformer  IC={transformer_scores["IC"]:+.4f}  '
      f'dir_acc={transformer_scores["dir_acc"]:.3f}  '
      f'sharpe={transformer_scores["sharpe"]:+.3f}')

Training transformer (134,785 params) for 40 epochs...
  ep  10/40  loss=0.00017
  ep  20/40  loss=0.00015
  ep  30/40  loss=0.00012
  ep  40/40  loss=0.00010
Done in 38.1s

Transformer  IC=+0.0523  dir_acc=0.524  sharpe=+0.177


## 5. Model 3 — Multi-Scale LSTM Ensemble

Three separate LSTM networks, each trained on a different lookback window:
- **Weekly** (5 days) — captures short-term momentum & reversal
- **Monthly** (21 days) — captures trend & carry
- **Quarterly** (63 days) — captures regime & factor rotation

In [7]:
class ScaleLSTM(nn.Module):
    def __init__(self, n_features: int, hidden: int = 64, n_layers: int = 2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features, hidden_size=hidden,
            num_layers=n_layers, batch_first=True,
            dropout=DROPOUT if n_layers > 1 else 0.0
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden, 1)
        )

    def forward(self, x):           # x: (B, T, F)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


SCALES = {
    'weekly':    5,
    'monthly':   21,
    'quarterly': 63,
}

scale_preds  = {}   # name → pd.Series on test index
scale_scores = {}
scale_models = {}

for scale_name, lookback in SCALES.items():
    SEP = '='*55
    print(f'\n{SEP}')
    print(f' Scale: {scale_name.upper()}  (lookback={lookback}d)')
    print(SEP)

    Xs_tr_s, ys_tr_s, idx_tr_s, Xs_te_s, ys_te_s, idx_te_s = build_sequence_dataset(
        df, SIGNAL_COLS, TARGET, lookback, TRAIN_END, TEST_START, scaler
    )
    print(f'Dataset: train={Xs_tr_s.shape}  test={Xs_te_s.shape}')

    mdl, pred = train_seq_model(
        ScaleLSTM(N_FEATURES),
        Xs_tr_s, ys_tr_s, Xs_te_s, idx_te_s,
        epochs=SCALE_EPOCHS, name=f'nn_{scale_name}'
    )
    sc = score(f'NN-{scale_name.capitalize()}', pred, y_test)
    scale_preds[scale_name]  = pred
    scale_scores[scale_name] = sc
    scale_models[scale_name] = mdl
    print(f'IC={sc["IC"]:+.4f}  dir_acc={sc["dir_acc"]:.3f}  sharpe={sc["sharpe"]:+.3f}')

## 6. Meta-Learner — IC-Weighted Combination

A Ridge regression trained on the **validation** portion of the training set uses all 5 model predictions as features and learns optimal combination weights. On the test set the weights are fixed.

In [ ]:
# ── Collect all train predictions by re-running inference on train set ─────────
# MLP train preds
mlp_model.eval()
with torch.no_grad():
    mlp_train_pred = mlp_model(
        torch.tensor(X_train_s, dtype=torch.float32).to(DEVICE)
    ).cpu().numpy()
mlp_train_series = pd.Series(mlp_train_pred, index=y_train.index, name='mlp')

# Transformer + scale LSTMs: use 20% of train as val (time-ordered)
# Build sequence train preds for transformer
transformer_model.eval()
trans_tr_preds = []
with torch.no_grad():
    for i in range(0, len(Xs_tr), BATCH_SIZE):
        xb = torch.tensor(Xs_tr[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
        trans_tr_preds.append(transformer_model(xb).cpu().numpy())
trans_train_series = pd.Series(np.concatenate(trans_tr_preds), index=idx_tr, name='transformer')

# Scale LSTM train preds
scale_train_preds = {}
for scale_name, lookback in SCALES.items():
    Xs_tr_s, ys_tr_s, idx_tr_s, _, _, _ = build_sequence_dataset(
        df, SIGNAL_COLS, TARGET, lookback, TRAIN_END, TEST_START, scaler
    )
    mdl = scale_models[scale_name].eval()
    preds_tr = []
    with torch.no_grad():
        for i in range(0, len(Xs_tr_s), BATCH_SIZE):
            xb = torch.tensor(Xs_tr_s[i:i+BATCH_SIZE], dtype=torch.float32).to(DEVICE)
            preds_tr.append(mdl(xb).cpu().numpy())
    scale_train_preds[scale_name] = pd.Series(
        np.concatenate(preds_tr), index=idx_tr_s, name=f'nn_{scale_name}'
    )
print('Train predictions collected.')

# ── Align all train predictions on a common index ─────────────────────────────
meta_train = pd.DataFrame({
    'mlp':         mlp_train_series,
    'transformer': trans_train_series,
    **{f'nn_{k}': v for k, v in scale_train_preds.items()},
})
meta_train = meta_train.join(y_train.rename('y'), how='inner').dropna()

# ── Train Ridge meta-learner ───────────────────────────────────────────────────
feat_cols = [c for c in meta_train.columns if c != 'y']
meta_X = meta_train[feat_cols].values
meta_y = meta_train['y'].values

meta_learner = Ridge(alpha=1.0, fit_intercept=False)
meta_learner.fit(meta_X, meta_y)

print('\nMeta-learner weights:')
for name, w in zip(feat_cols, meta_learner.coef_):
    print(f'  {name:15s}: {w:+.4f}')

# ── Generate meta-learner test predictions ─────────────────────────────────────
meta_test = pd.DataFrame({
    'mlp':         mlp_pred,
    'transformer': transformer_pred,
    **{f'nn_{k}': v for k, v in scale_preds.items()},
}).dropna()

meta_pred = pd.Series(
    meta_learner.predict(meta_test[feat_cols].values),
    index=meta_test.index,
    name='meta'
)
meta_scores = score('MetaLearner', meta_pred, y_test)
print(f'\nMeta-learner  IC={meta_scores["IC"]:+.4f}  '
      f'dir_acc={meta_scores["dir_acc"]:.3f}  '
      f'sharpe={meta_scores["sharpe"]:+.3f}')

## 7. Results — Compare All Models

In [ ]:
all_scores = [
    mlp_scores,
    transformer_scores,
    *[score(f'NN-{k.capitalize()}', v, y_test) for k, v in scale_preds.items()],
    meta_scores,
]
results_df = pd.DataFrame(all_scores).set_index('model').sort_values('IC', ascending=False)
print('\nModel comparison (test set):')
print(results_df.round(4).to_string())

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col, title in zip(axes, ['IC', 'dir_acc', 'sharpe'],
                           ['Spearman IC', 'Directional Accuracy', 'Long-Short Sharpe']):
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in results_df[col]]
    results_df[col].plot(kind='barh', ax=ax, color=colors)
    ax.axvline(0, color='black', linewidth=0.8)
    ax.set_title(title)
    ax.set_xlabel(col)
    ax.grid(axis='x', alpha=0.3)

plt.suptitle('Stage 2.2 — Multi-Scale Neural Ensemble', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, 'neural_ensemble_chart.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved.')

## 8. Rolling IC — Scale Comparison Over Time

Shows which time scale is most predictive in different market regimes.

In [ ]:
TEST_DATES = sorted(y_test.index.get_level_values('timestamp').unique())

all_preds_map = {
    'MLP':        mlp_pred,
    'Transformer':transformer_pred,
    'NN-Weekly':  scale_preds['weekly'],
    'NN-Monthly': scale_preds['monthly'],
    'NN-Quarterly':scale_preds['quarterly'],
    'Meta':       meta_pred,
}

daily_ics = []
for d in TEST_DATES:
    actual_d = y_test.xs(d, level='timestamp') if d in y_test.index.get_level_values('timestamp') else None
    if actual_d is None or len(actual_d) < 5:
        continue
    row = {'date': d}
    for mname, pseries in all_preds_map.items():
        try:
            pred_d = pseries.xs(d, level='timestamp')
            row[mname] = spearman_ic(pred_d, actual_d)
        except Exception:
            pass
    daily_ics.append(row)

ic_df = pd.DataFrame(daily_ics).set_index('date')
ic_roll = ic_df.rolling(20, min_periods=5).mean()

fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.tab10.colors
for i, col in enumerate(ic_roll.columns):
    lw = 2.5 if col == 'Meta' else 1.3
    ax.plot(ic_roll.index, ic_roll[col], label=col,
            linewidth=lw, color=colors[i % len(colors)])
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_ylabel('20-day rolling IC')
ax.set_title('Rolling IC by Time Scale — which horizon is most predictive?')
ax.legend(fontsize=8, ncol=3, loc='upper right')
ax.grid(alpha=0.3)
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
fig.autofmt_xdate()
plt.tight_layout()
plt.savefig(os.path.join(WORKSPACE, 'neural_rolling_ic.png'), dpi=150, bbox_inches='tight')
plt.show()

## 9. Save Predictions

In [ ]:
# Combine with existing Stage 2 alpha_mu
alpha_mu_existing = pd.read_parquet(os.path.join(DATA_DIR, 'alpha_mu.parquet'))

neural_preds = pd.DataFrame({
    'nn_mlp':        mlp_pred,
    'nn_transformer':transformer_pred,
    'nn_weekly':     scale_preds['weekly'],
    'nn_monthly':    scale_preds['monthly'],
    'nn_quarterly':  scale_preds['quarterly'],
    'nn_meta':       meta_pred,
})

alpha_mu_neural = alpha_mu_existing.join(neural_preds, how='outer')
alpha_mu_neural.to_parquet(os.path.join(DATA_DIR, 'alpha_mu_neural.parquet'))
print(f'Saved alpha_mu_neural.parquet: {alpha_mu_neural.shape}')
print(f'Columns: {list(alpha_mu_neural.columns)}')

# Save models
import pickle
torch.save(mlp_model.state_dict(), os.path.join(MODELS_DIR, 'nn_mlp.pt'))
torch.save(transformer_model.state_dict(), os.path.join(MODELS_DIR, 'nn_transformer.pt'))
for sname, smdl in scale_models.items():
    torch.save(smdl.state_dict(), os.path.join(MODELS_DIR, f'nn_{sname}.pt'))
with open(os.path.join(MODELS_DIR, 'nn_meta_learner.pkl'), 'wb') as f:
    pickle.dump(meta_learner, f)
with open(os.path.join(MODELS_DIR, 'nn_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)

print('\nModel artifacts saved:')
for fname in ['nn_mlp.pt', 'nn_transformer.pt', 'nn_weekly.pt',
              'nn_monthly.pt', 'nn_quarterly.pt', 'nn_meta_learner.pkl']:
    print(f'  {MODELS_DIR}/{fname}')

## 10. Stage 2.2 Readiness Summary

In [ ]:
print('=' * 68)
print(' STAGE 2.2 — MULTI-SCALE NEURAL ENSEMBLE — READINESS SUMMARY')
print('=' * 68)

print('\nModel performance (test set):')
print(results_df.round(4).to_string())

best_model = results_df['IC'].abs().idxmax()
print(f'\nBest neural model by |IC|: {best_model}  '
      f'(IC={results_df.loc[best_model, "IC"]:+.4f})')

print(f'\nOutputs:')
print(f'  {DATA_DIR}/alpha_mu_neural.parquet')
print(f'  {WORKSPACE}/neural_ensemble_chart.png')
print(f'  {WORKSPACE}/neural_rolling_ic.png')
print(f'  {MODELS_DIR}/nn_*.pt  +  nn_meta_learner.pkl')

print(f'\nReady to merge neural signals into Stage 3 inference.')
print('=' * 68)